# 🧪 W10-D2 Audit 与 Trace：可查询的执行证据

> 配套阅读：同名 `.md`。本 notebook 只用小规模、可重复的模拟来验证核心治理约束。

**实验目标：** 模拟 Trace 的跨服务关联、按类型/状态查询，以及与扁平 Audit 的关联。


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

from dataclasses import dataclass, asdict
import uuid

trace_id = "trace-" + uuid.uuid4().hex[:8]
@dataclass
class ExecutionSpan:
    trace_id: str; span_id: str; parent_span_id: str | None; kind: str; duration_ms: int; status: str; tenant_id: int

rows = [
    ExecutionSpan(trace_id, "root", None, "channel_dispatch", 910, "ok", 7),
    ExecutionSpan(trace_id, "rag", "root", "rag_retrieval", 260, "ok", 7),
    ExecutionSpan(trace_id, "llm", "root", "llm", 620, "timeout", 7),
    ExecutionSpan(trace_id, "reply", "root", "adapter", 12, "ok", 7),
]
print("trace_id:", trace_id)
for row in rows: print(asdict(row))


In [ ]:
# “SQL 风格”结构化过滤：不是 grep 文本，而是按语义字段找失败调用。
failed_llm = [s for s in rows if s.kind == "llm" and s.status != "ok"]
tenant_spans = [s for s in rows if s.tenant_id == 7]
print("失败的 LLM span:", [asdict(s) for s in failed_llm])
print("租户 7 的总耗时:", sum(s.duration_ms for s in tenant_spans), "ms")

# Trace payload 可以存完整 I/O；span 行只放可聚合的摘要。
payload_store = {"llm": {"input": "退款政策是什么？", "output": "<timeout>"}}
print("payload side table:", payload_store["llm"])


In [ ]:
kinds = sorted({s.kind for s in rows})
ok_counts = [sum(s.kind == k and s.status == "ok" for s in rows) for k in kinds]
err_counts = [sum(s.kind == k and s.status != "ok" for s in rows) for k in kinds]
x = np.arange(len(kinds))
plt.figure(figsize=(7, 3.5))
plt.bar(x - .18, ok_counts, .36, label="ok", color="#59A14F")
plt.bar(x + .18, err_counts, .36, label="non-ok", color="#E15759")
plt.xticks(x, kinds); plt.ylabel("span 数"); plt.title("按 SpanKind/状态聚合")
plt.legend(); plt.tight_layout(); plt.show()

audit_event = {"actor_id": "u-42", "action": "skill_release:invoke", "result": "failed", "trace_id": trace_id}
print("以 trace_id 关联的 Audit：", audit_event)
